# Inteligência Artificial em Séries Temporais de Saúde

Este notebook compara um classificador tradicional com um modelo neuromórfico inspirando-se em neurônios LIF (Leaky Integrate-and-Fire).

O objetivo é analisar padrões em janelas temporais de sinais vitais e discutir vantagens e limitações de cada abordagem.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Geração de sinais sintéticos

Como o foco é didático e comparativo, vamos gerar séries temporais representando padrões regulares e irregulares, além de ruído e variações que simulam sinais vitais.

In [ ]:
def generate_window(length=128, label=0, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)

    t = np.linspace(0, 1, length)
    if label == 0:
        signal = np.sin(2 * np.pi * 3 * t)
        signal += 0.05 * np.sin(2 * np.pi * 9 * t)
        signal += rng.normal(0, 0.08, size=length)
    else:
        signal = np.sin(2 * np.pi * 3 * t + 0.6)
        signal += 0.60 * np.sin(2 * np.pi * 12 * t)
        signal += rng.normal(0, 0.18, size=length)
        signal += np.linspace(0.0, 0.8, length)
        signal = np.clip(signal, -2.0, 2.0)
    return signal.astype(float)

def generate_dataset(n_samples=600, window_length=128, seed=42):
    rng = np.random.default_rng(seed)
    windows = []
    labels = []

    for _ in range(n_samples):
        label = int(rng.random() < 0.5)
        window = generate_window(window_length, label=label, rng=rng)
        windows.append(window)
        labels.append(label)

    X = np.vstack(windows)
    y = np.asarray(labels, dtype=int)
    return X, y

X, y = generate_dataset(n_samples=800, window_length=128, seed=42)
X.shape, y.shape

## 2. Feature engineering

A regressão logística precisa de um vetor de features estáticas. Em contraste, o modelo LIF é inspirado na dinâmica temporal de neurônios.

In [ ]:
def statistical_features(window):
    values = np.asarray(window, dtype=float)
    return np.array([
        values.mean(),
        values.std(),
        values.min(),
        values.max(),
        np.sqrt(np.mean(np.square(values))),
        np.mean(np.abs(np.diff(values))),
        np.abs(np.diff(values)).mean(),
        np.ptp(values),
    ], dtype=float)

def lif_feature_extractor(window, threshold=0.75, decay=0.85):
    signal = np.asarray(window, dtype=float)
    membrane = 0.0
    spikes = 0
    spike_times = []
    max_membrane = 0.0

    for value in signal:
        membrane = decay * membrane + value
        max_membrane = max(max_membrane, membrane)
        if membrane > threshold:
            spikes += 1
            spike_times.append(1)
            membrane = 0.0
        else:
            spike_times.append(0)

    firing_rate = spikes / len(signal)
    return np.array([
        spikes,
        firing_rate,
        np.mean(spike_times),
        max_membrane,
        np.abs(signal).mean(),
        np.std(signal),
    ], dtype=float)

def build_feature_matrix(X, mode='statistical'):
    features = []
    for sample in X:
        if mode == 'statistical':
            features.append(statistical_features(sample))
        elif mode == 'lif':
            features.append(lif_feature_extractor(sample))
        else:
            raise ValueError(f'Modo inválido: {mode}')
    return np.vstack(features)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
X_train_stat = build_feature_matrix(X_train, mode='statistical')
X_test_stat = build_feature_matrix(X_test, mode='statistical')
X_train_lif = build_feature_matrix(X_train, mode='lif')
X_test_lif = build_feature_matrix(X_test, mode='lif')

X_train_stat.shape, X_train_lif.shape

In [ ]:
def fit_and_score(X_train, y_train, X_test, y_test, label):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = {
        'modelo': label,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, pred).tolist(),
    }
    return metrics

metrics_logistic = fit_and_score(X_train_stat, y_train, X_test_stat, y_test, 'regressao_logistica')
metrics_lif = fit_and_score(X_train_lif, y_train, X_test_lif, y_test, 'lif')
metrics_logistic, metrics_lif

## 3. Comparação dos resultados

Abaixo, observamos a diferença entre o baseline clássico e a dinâmica bioinspirada do modelo LIF.

In [ ]:
sns.set_theme(style='whitegrid')
comparison = pd.DataFrame([
    metrics_logistic,
    metrics_lif
]).set_index('modelo')

comparison[['accuracy', 'precision', 'recall', 'f1']].plot(kind='bar', figsize=(10, 5), alpha=0.9)
plt.title('Comparação de desempenho')
plt.ylabel('Métrica')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

comparison

## 4. Discussão crítica

A regressão logística costuma ter desempenho superior em problemas bem estruturados e com features relevantes. Já o modelo LIF pode trazer vantagens quando a dinâmica temporal e os eventos discretos são centrais.

A principal limitação do LIF é a sensibilidade aos parâmetros e a menor interpretabilidade direta. Em dados clínicos reais, a robustez e a qualidade do rótulo são críticas. Portanto, a comparação deve ser feita de forma crítica, não absoluta.